## 01 

### N-gram
- 언어 모델이란?
    - 문장이 얼마나 자연스러운지 확률로 수치화
    - 이전 단어들이 주어졌을 때 다음 단어 예측하는 시스템

#### N-gram
- 다음 단어를 예측하기 위해 앞 몇 개의 단어를 참고할 것이지 결정하는 방식
- 종류            참고 단어 수    
Unigram(1-gram) : 0개 (이전 문맥 무시)  
Bigram(2-gram)  : 앞 1개  
Trigram(3-gram) : 앞 2개 

#### N-gram의 3가지 한계
- 1. 말뭉치(Corpus) 편향성
    - 모델은 학습 데이터 속 세상만 인식 => 학습 데이터에 없는 패턴이 오면 확률 = 0 => 희소 문제(Sparsity Problem)
- 2. 토큰화 방식 종속성
    - 어절 단위 : [나는, 오늘, 밥을, 먹었다] -> 4개 토큰
    - 형태소 단위 : [나, 는, 오늘, 밥, 을, 먹, 었, 다] -> 8개 토큰
    - 한국어는 교착어라 형태소 분석이 필수
- 3. 장기 의존성(Long-term Dependency) 문제
    - "그 프랑스 출신 요리사는 ... 고향을 그리워하며 [ ] 로 중얼거린다"
    - Trigram은 앞 2단어만 봐서 멀리 있는 "프랑스"정보를 놓침
    - N을 늘리면? -> 희소 문제 발생(패턴이 데이터에 없음)

#### 특수 토큰 BOS / EOS
- 문장 경계를 모델에게 알려주기 위해 삽입
- <`s`>  : 문장 시작 (Begin of Sentence)
- </s> : 문장 끝 (End od Sentence)
    - 원본 : 나는 밥을 먹었다
    - 전처리 : <`s`> 나는 밥을 먹었다 </s>

In [1]:
# step 1. 말뭉치 전처리 (BOS / EOS 부착)
raw_corpus = [
    '나는 오늘 파이토치 학습을 한다',
    '나는 내일 자연어 처리를 한다',
    '오늘 파이토치 학습은 매우 즐겁다',
    '파이토치 코딩은 매우 재미있다'
]

def process_corpus(corpus):
    processed = []
    for sentence in corpus:
        processed.append(['<s>'] + sentence.split() + ['</s>'])
    return processed

tokenized_corpus = process_corpus(raw_corpus)
print(tokenized_corpus)

[['<s>', '나는', '오늘', '파이토치', '학습을', '한다', '</s>'], ['<s>', '나는', '내일', '자연어', '처리를', '한다', '</s>'], ['<s>', '오늘', '파이토치', '학습은', '매우', '즐겁다', '</s>'], ['<s>', '파이토치', '코딩은', '매우', '재미있다', '</s>']]


In [2]:
# step 2. N-gram 추출기 (슬라이딩 윈도우)
from collections import Counter

def extract_ngrams(tokenized_corpus, n):
    ngram_counts = Counter()
    for tokens in tokenized_corpus:
        if len(tokens) < n:     # 문장이 너무 짧으면 건너뜀
            continue
        for i in range(len(tokens) - n + 1): # 슬라이딩 윈도우
            ngram = tuple(tokens[i : i + n])  # 튜플로 변환 (해시 가능)
            ngram_counts[ngram] += 1
    return ngram_counts

In [3]:
unigram = extract_ngrams(tokenized_corpus, 1)
bigram  = extract_ngrams(tokenized_corpus, 2)
trigram = extract_ngrams(tokenized_corpus, 3)

print("\n=== Unigram Top 5 ===")
print(unigram.most_common(5))

print("\n=== Bigram Top 5 ===")
print(bigram.most_common(5))


=== Unigram Top 5 ===
[(('<s>',), 4), (('</s>',), 4), (('파이토치',), 3), (('나는',), 2), (('오늘',), 2)]

=== Bigram Top 5 ===
[(('<s>', '나는'), 2), (('오늘', '파이토치'), 2), (('한다', '</s>'), 2), (('나는', '오늘'), 1), (('파이토치', '학습을'), 1)]


In [4]:
#==== range(len(tokens) - n + 1) ====
# 토큰이 4개이고 n=2라면 만들 수 있는 bigram은 3개 (4 - 2 + 1)
# 마지막 ngram이 범위를 벗어나지 않도록 조정

#==== Counter vs dict 차이 ====
# Counter는 없는 키를 조회하면 자동으로 0 반환 -> 초기화 불필요
# 일반 dict
# d = {}
# d['나는'] += 1  # KeyError! '나는' 키가 없어서 에러
# Counter
# c = Counter()
# c['나는'] += 1  # 없는 키는 자동으로 0으로 시작

#==== 해시 가능(Hashable) ====
# 딕셔너리는 내부적으로 키를 [숫자(해시값) => 어떤 고정된 숫자] 로 변환해서 저장
# 리스트는 값이 바뀔 수 있어서(mutable) 해시값이 변할 수 있음
# 튜플은 바꿀 수 없음(immutable) => 해시값 고정 => 키로 안전

In [5]:
# step 3. 다음 단어 예측 시뮬레이션
def simulate_prediction(ngram_counts, context_words):
    context_len = len(context_words)
    found = False

    for ngram, count in ngram_counts.items():
        if list(ngram[:context_len]) == context_words:
            next_word = ngram[context_len]
            print(f"-->예측단어 : {next_word} 등장횟수 : {count}번")
            found = True
    
    if not found:
        print("데이터베이스에 해당 문맥이 존재하지 않습니다.(희소문제 발생)")

# ngram[:context_len] : 튜플 슬라이싱. n-gram의 앞부분만 잘라냄
# list(ngram[:context_len]) == context_words : 튜플을 리스트 변환후 비교(context_words가 리스트라서)
# ngram[context_len] : context바로 다음 위치 = 예측 후보 단어
# fount = False : 플래그 -> 끝까지 매칭이 없으면 희소문제 메세지 출력

In [6]:
print("\n=== [Bigram] '나는' 다음 단어 ===")
simulate_prediction(bigram, ['나는'])

print("\n=== [Trigram] '나는 오늘' 다음 단어 ===")
simulate_prediction(trigram, ['나는', '오늘'])

print("\n=== [Trigram] 희소 문제 테스트 ===")
simulate_prediction(trigram, ['파이토치', '처리를'])

# ── 버그 수정: four_gram과 trigram 분리 ──────────────────────
four_gram = extract_ngrams(tokenized_corpus, 4)  # trigram 덮어쓰기 없음!

print("\n=== [4-gram] '나는 오늘 파이토치' 다음 단어 ===")
simulate_prediction(four_gram, ['나는', '오늘', '파이토치'])


=== [Bigram] '나는' 다음 단어 ===
-->예측단어 : 오늘 등장횟수 : 1번
-->예측단어 : 내일 등장횟수 : 1번

=== [Trigram] '나는 오늘' 다음 단어 ===
-->예측단어 : 파이토치 등장횟수 : 1번

=== [Trigram] 희소 문제 테스트 ===
데이터베이스에 해당 문맥이 존재하지 않습니다.(희소문제 발생)

=== [4-gram] '나는 오늘 파이토치' 다음 단어 ===
-->예측단어 : 학습을 등장횟수 : 1번


## 02

### 조건부 확률
- 언어 모델이 확률을 계산하는 가장 기본 원리  
P( B | A ) = Count(A->B) / Count(A)  

-분자 : A 다음에 B가 등장한 횟수  
-분모 : A가 등장한 전체 횟수  

소설책에서 '맛잇는'이 10번 등장  
그 중 '맛있는' 바로 뒤에 '피자를'이 7번 등장  
=> P(피자를 | 맛있는) = 7 / 10 = 0.7 (70% 확률로 예측)

### 연쇄 법칙(Chain Rule)
- 문장 전체의 확률은 단어 하나하나의 조건부 확률을 곱해서 계산
P(나는 오늘 밥을 먹었다)  
= P(나는)   
× P(오늘 | 나는)  
× P(밥을 | 나는, 오늘)  
× P(먹었다 | 나는, 오늘, 밥을)  

- 문제점 : 문장이 길어질수록 조건부 항이 무한이 길어짐
- 현실에서 계산이 불가능해짐 -> 마르코프 가정으로 해결

### 마르코프 가정 (Markov Assumption)
- 연쇄 법칙의 문제를 해결하기 위한 실무적 타협안
- "현재 단어는 바로 앞 N-1개 단어에만 의존한다"고 가정하고 나머지는 잘라버림
    - 연쇄 법칙 원래:  
    P(먹었다 | 나는, 오늘, 밥을) → 앞 단어 전부 필요

    - Bigram 가정 후:  
    P(먹었다 | 밥을) → 직전 1개만!

    - Trigram 가정 후:  
    P(먹었다 | 오늘, 밥을) → 직전 2개만!
- 계산은 훨씬 쉬워지지만 앞의 정보를 버린다는 손실이 생김

### 최대 우도 추정 (MLE)
- N-gram확률을 구하는 정석 방법(Bigram 기준)
- 말뭉치에서 등장한 빈도(Count)를 그대로 확률로 사용하는 방법
- 공식 : P(다음단어 | 현재단어) = Count(현재->다음) / Count(현재)   
"나는"이 3번 등장  
"나는 오늘"이 2번 등장  
→ P(오늘|나는) = 2/3 = 0.667  

In [7]:
# step 1. 클래스 구조(__init__)
class MLELanguageModel:
    def __init__(self, n):
        self.n = n
        self.ngram_counts = Counter()   # 분자 : (context + target) 빈도
        self.context_counts = Counter() # 분모 : context 빈도
# Counter를 2개 따로 => 분모/ 분자를 분리해서 저장
    
    # step 2. train() 메서드
    def train(self, corpus):
        for sentence in corpus:
            tokens = ['<s>'] + sentence.split() + ['</s>']
            for i in range(len(tokens) - self.n + 1):
                ngram = tuple(tokens[i : i + self.n])       # N개
                context = tuple(tokens[i : i + self.n - 1]) # N-1개
                self.ngram_counts[ngram] += 1
                self.context_counts[context] += 1
    
    # step 3. get_probability() 메서드
    def get_probability(self, context, target):
        context_tuple = tuple(context)
        ngram_tuple = tuple(context) + (target,)    # 튜플을 만들려면 ,(쉼표) 필요, 없으면 그냥 문자열

        numerator = self.ngram_counts.get(ngram_tuple, 0)
        denominator = self.context_counts.get(context_tuple, 0)

        if denominator == 0:
            return 0.0
        return numerator / denominator

In [8]:
# ── 2. 학습 ──────────────────────────────────────────────────
train_corpus = [
    '나는 오늘 파이토치 학습을 한다',
    '나는 내일 자연어 처리를 한다',
    '나는 오늘 머신러닝 논문을 읽는다'
]

bigram_model = MLELanguageModel(n=2)
bigram_model.train(train_corpus)


# ── 3. 내부 카운터 확인 ──────────────────────────────────────
print("=== ngram_counts (분자) ===")
print(bigram_model.ngram_counts)

print("\n=== context_counts (분모) ===")
print(bigram_model.context_counts)


# ── 4. 확률 계산 및 검증 ─────────────────────────────────────
print("\n=== 확률 계산 ===")
p = bigram_model.get_probability(['나는'], '오늘')
print(f"P(오늘 | 나는) = {p:.4f}")  # 기댓값: 2/3 = 0.6667

p2 = bigram_model.get_probability(['나는'], '내일')
print(f"P(내일 | 나는) = {p2:.4f}")  # 기댓값: 1/3 = 0.3333

print(f"\n합계 검증: {p + p2:.4f} (1.0이 되어야 함)")  # 확률 합 = 1

=== ngram_counts (분자) ===
Counter({('<s>', '나는'): 3, ('나는', '오늘'): 2, ('한다', '</s>'): 2, ('오늘', '파이토치'): 1, ('파이토치', '학습을'): 1, ('학습을', '한다'): 1, ('나는', '내일'): 1, ('내일', '자연어'): 1, ('자연어', '처리를'): 1, ('처리를', '한다'): 1, ('오늘', '머신러닝'): 1, ('머신러닝', '논문을'): 1, ('논문을', '읽는다'): 1, ('읽는다', '</s>'): 1})

=== context_counts (분모) ===
Counter({('<s>',): 3, ('나는',): 3, ('오늘',): 2, ('한다',): 2, ('파이토치',): 1, ('학습을',): 1, ('내일',): 1, ('자연어',): 1, ('처리를',): 1, ('머신러닝',): 1, ('논문을',): 1, ('읽는다',): 1})

=== 확률 계산 ===
P(오늘 | 나는) = 0.6667
P(내일 | 나는) = 0.3333

합계 검증: 1.0000 (1.0이 되어야 함)


## 03

### PPL이란? "얼마나 당황스러운가"
- Perplecxity(퍼플렉서티)의 사전적 의미는 '당혹감', '복잡함'
- 언어 모델 입장에서 "다음 단어를 예측할 때 얼마나 갈팡질팡하는가"를 수치로 나타낸것
    - 낮은 PPL : 모델이 다음 단어를 확신 => 좋은 모델
    - 높은 PPL : 모델이 어떤 단어가 올지 몰라 당황 => 나쁜 모델  
        PPL               값의미    
        PPL = 2         : 2지선다(O/X) 수준 → 거의 확신  
        PPL = 10        : 10지선다 → 양호P  
        PL = 100        : 100지선다 → 매우 혼란  
        PPL = 전체 어휘 수 : 완전 무작위 추측 → 최악  

### PPL의 수학적 정의
PPL(W) = P(w₁, w₂, ..., wₙ)^(-1/N)  
       = N√(1 / P(w₁, w₂, ..., wₙ))  
P(W): 문장 전체 확률  
N   : 문장 길이 (단어 수)  
- 역수를 취하는 이유  

확률 P(W)가 높을수록 → 좋은 모델  
근데 확률은 0~1 사이의 작은 값  

역수(1/P) 취하면 →  
  확률 높음 → 역수 작음 → PPL 낮음   
  확률 낮음 → 역수 큼   → PPL 높음   

- N제곱근을 취하는 이유

문장이 길수록 확률이 작아짐 → PPL도 커짐  
N제곱근으로 문장 길이만큼 정규화  
→ 긴 문장 vs 짧은 문장 공정하게 비교 가능  

### 언더플로우 (Underflow)
- PPL을 수식 그대로 계산하면 문제가 생김
    - 단어 하나의 확률 = 0.1
    - 100개 단어 문장 확률 = 0.1^100 = 10^(-100)
- 컴퓨터의 한계
    - Python float64 표현 가능 범위를 넘어감
    - 컴퓨터가 표현 못하고 0으로 증발
    - PPL = (1 / 0.0) ^ (1/N) => ZeroDivisionError

In [9]:
joint = 1.0
for _ in range(1000):
    joint *= 0.1

print(joint)

0.0


### 로그(log) 변환으로 해결
- log(a x b) = log(a) + log(b)
- 곱셈 => 덧셈으로 변환

#### 원래 수식 (언더플로우 위험)
PPL = (1 / P(w₁) × P(w₂) × ... × P(wₙ))^(1/N)

#### 로그 취하면 곱셈 → 덧셈
log PPL = -1/N × (log P(w₁) + log P(w₂) + ... + log P(wₙ))

#### exp로 복원
PPL = exp( -1/N × Σ log P(wᵢ) )

### 수치 증명
- 일반 수식
1. P(W) = 0.1 × 0.1 = 0.01
2. PPL  = (1 / 0.01)^(1/2)
        = 100^0.5
        = 10.0
- 로그 수식
1. log(0.1) ≈ -2.3026
2. 로그 합산: -2.3026 + (-2.3026) = -4.6052
3. 평균:      -1/2 × (-4.6052) = 2.3026
4. exp 복원:  exp(2.3026) ≈ 10.0

- Cross-Entropy와의 관계  
로그 PPL 수식의 내부:  
-1/N × Σ log P(wᵢ) = Cross-Entropy Loss  

즉, PPL = exp(Cross-Entropy Loss)  

딥러닝에서 loss가 낮아질수록  
PPL도 낮아지는 이유가 바로 이것!  

In [10]:
# step 1. PPL 일반 수식 계산
def calc_ppl_normal(probs):
    N = len(probs)
    joint_prob = 1.0
    for p in probs:
        joint_prob *= p    # 확률 곱셈
    ppl = (1 / joint_prob) ** (1 / N)   # 역수의 N제곱근
    return ppl

In [11]:
# step 2. PPL 로그 수식 계산
def calc_ppl_log(probs):
    N = len(probs)
    log_sum = 0.0
    for p in probs:
        log_sum += math.log(p)          # 곱 대신 로그 합산
    ppl = math.exp(-1 / N * log_sum)    # exp로 복원
    return ppl

In [12]:
import math
# step 3. 두 방식 비교
probs = [0.1, 0.1]

ppl_normal = calc_ppl_normal(probs)
ppl_log = calc_ppl_log(probs)

print(f"일반 수식 PPL : {ppl_normal:.4f}")
print(f"로그 수식 PPL : {ppl_log:.4f}")
print(f"두 결과 일치  : {abs(ppl_normal - ppl_log) < 1e-9}")    # 부동소수점 연산은 미세한 오차가 생길 수 있기 때문에

일반 수식 PPL : 10.0000
로그 수식 PPL : 10.0000
두 결과 일치  : True


In [13]:
# step 4. 언더플로우 시뮬레이션
probs_long = [0.1] * 1000   # 1000 단어 문장

# 일반 수식 -> 언더플로우
joint = 1.0
for p in probs_long:
    joint *= p
print(f"일반 곱셈 결과 : {joint}")  # 0.0 으로 증발

# 로그 수식 -> 안전
log_sum = sum(math.log(p) for p in probs_long)  # 제너레이터 표현식 : 필요할 때마다 값을 하나씩 생성
ppl_safe = math.exp(-1 / len(probs_long) * log_sum)
print(f"로그 수식 PPL  : {ppl_safe:.4f}")   # 정상 계산


일반 곱셈 결과 : 0.0
로그 수식 PPL  : 10.0000


In [15]:
# step 5. 문맥에 따른 PPL 변화
# 문맥 없음 -> 선택지 많음 -> 높은 PPL
probs_no_context = [0.01, 0.01, 0.01]   # 각 단어 확률 1%
# 문맥 풍부 -> 선택지 적음 -> 낮은 PPL
probs_with_context = [0.8, 0.7, 0.9]   # 각 단어 확률 70~90%

print(f"문맥 없음 PPL  : {calc_ppl_log(probs_no_context):.4f}")
print(f"문맥 풍부 PPL  : {calc_ppl_log(probs_with_context):.4f}")

문맥 없음 PPL  : 100.0000
문맥 풍부 PPL  : 1.2566


## 04

### Laplace Smoothing
- 학습 데이터에 "오늘 커피를" 이라는 조합이 없다면   
-> 문장 전체 확률 = 0 ("이 문장은 존재 불가능" 으로 판단)

- 해결책 : Laplace Smoothing(Add-1)  
    - P(wₙ | wₙ₋₁) = (Count(wₙ₋₁, wₙ) + 1) / (Count(wₙ₋₁) + V)
-> 분자에 + 1 : 한 번도 안 나온 조합도 최소 1번으로 처리  
-> 분모에 + V : 모든 단어에 +1 줬으니 분모도 V만큼 증가  

### 감성 분류 원리 (Naive Bayes)
- N-gram 언어 모델로 텍스트 분류
- 핵심 아이디어
    - 긍정 리뷰들로만 학습 -> 긍정 모델
    - 부정 리뷰들로만 학습 -> 부정 모델
- 새로운 리뷰가 들어오면
    - 긍정 모델에서 이 문장이 나올 확률 계산
    - 부정 모델에서 이 문장이 나올 확률 계산
    - 둘 중 높은 쪽으로 분류

In [ ]:
from nltk.util import ngrams
from collections import defaultdict
import pandas as pd
import math

# ── 1. N-gram 모델 구축 (MLE) ────────────────────────────────
def build_ngram_model(corpus, n):
    model = defaultdict(lambda: defaultdict(lambda: 0))

    for sentence in corpus:
        tokens = sentence.split()
        ngram_list = list(ngrams(tokens, n,
                         pad_left=True, pad_right=True,
                         left_pad_symbol='<s>',
                         right_pad_symbol='</s>'))
        for ngram in ngram_list:
            context = ngram[:-1]
            target  = ngram[-1]
            model[context][target] += 1

    # 빈도 → 확률 변환 (MLE)
    for context in model:
        total = float(sum(model[context].values())) # model[context].values() → [1, 1, 1, 1] (4개 단어 각 1번씩)
        for target in model[context]:
            model[context][target] /= total
    return model

# defaultdict : 
# collections 모듈에 있는 자료형
# 존재하지 않는 키(key)에 접근했을 때 자동으로 기본값을 만들어주는 딕셔너리
# 일반 dict의 경우 없는 key가 들어오면 KeyError
# defaultdict(lambda : 0) : 없는 키는 자동으로 0으로 초기화
# model = defaultdict(lambda: defaultdict(lambda: 0)) # 중첩


# ── 2. 확률 조회 함수 ────────────────────────────────────────
def check_probability(model, context, target):
    prob = model[tuple(context)].get(target, 0) # 리스트 튜플로 변환 (딕셔너리 키로 사용하기 위해), 없는 키면 0 반환
    print(f"P('{target}' | {context}) = {prob:.4f}")


# ── 3. 다음 단어 예측 함수 ───────────────────────────────────
def generate_next_word(model, context):
    context = tuple(context)
    if context not in model:
        return "<Unknown>"
    next_word = max(model[context].items(), key=lambda x: x[1])[0]
    return next_word


# ── 4. Laplace Smoothing 모델 구축 ───────────────────────────
def build_ngram_model_with_smoothing(corpus, n):
    model = defaultdict(lambda: defaultdict(lambda: 0))
    vocab = set()

    for sentence in corpus:
        tokens = sentence.split()
        vocab.update(tokens)
        ngram_list = list(ngrams(tokens, n,
                         pad_left=True, pad_right=True,
                         left_pad_symbol='<s>',
                         right_pad_symbol='</s>'))
        for ngram in ngram_list:
            model[ngram[:-1]][ngram[-1]] += 1

    V = len(vocab) + 1  # </s> 포함

    smoothed_model = defaultdict(lambda: defaultdict(lambda: 0))
    for context in model:
        total = sum(model[context].values())
        for target in model[context]:
            smoothed_model[context][target] = (
                model[context][target] + 1) / (total + V)

    return smoothed_model, model, V


# ── 5. 문장 로그 확률 계산 ───────────────────────────────────
def calculate_sentence_log_prob(sentence, smoothed_model, raw_model, V, n):
    tokens = sentence.split()
    ngram_list = list(ngrams(tokens, n,
                     pad_left=True, pad_right=True,
                     left_pad_symbol='<s>',
                     right_pad_symbol='</s>'))
    log_prob = 0.0
    for ngram in ngram_list:
        context = ngram[:-1]
        target  = ngram[-1]

        if target in smoothed_model[context]:       # Case 1: 학습 시 본 조합
            prob = smoothed_model[context][target]
        elif context in raw_model:                  # Case 2: context만 아는 경우
            total = sum(raw_model[context].values())
            prob  = 1 / (total + V)
        else:                                       # Case 3: 완전 미지
            prob = 1 / V

        log_prob += math.log(prob)
    return log_prob


# ── 6. 감성 예측 함수 ────────────────────────────────────────
def predict_sentiment(sentence, models_info, n=2):
    best_class   = None
    max_log_prob = -float('inf')

    for label, (smoothed_model, raw_model, V) in models_info.items():
        log_prob = calculate_sentence_log_prob(
            sentence, smoothed_model, raw_model, V, n)
        if log_prob > max_log_prob:
            max_log_prob = log_prob
            best_class   = label
    return best_class


# ── 7. 데이터 준비 및 모델 학습 ─────────────────────────────
corpus = [
    "나는 오늘 맛있는 점심을 먹었다",
    "오늘 점심을 먹고 카페에 갔다",
    "나는 카페에 가서 맛있는 커피를 마셨다",
    "맛있는 점심을 먹는 것은 행복하다",
    "오늘 날씨는 정말 좋다",
    "나는 오늘 공부를 열심히 했다"
]

bigram_model  = build_ngram_model(corpus, 2)
trigram_model = build_ngram_model(corpus, 3)

print("=== Bigram 확률 조회 ===")
check_probability(bigram_model, ['오늘'], '점심을')
check_probability(bigram_model, ['나는'], '오늘')

print("\n=== Trigram 확률 조회 ===")
check_probability(trigram_model, ['맛있는', '점심을'], '먹었다')
check_probability(trigram_model, ['나는', '오늘'], '공부를')

print("\n=== 다음 단어 예측 ===")
print(f"'카페에' 다음: {generate_next_word(bigram_model, ['카페에'])}")
print(f"'맛있는 점심을' 다음: {generate_next_word(trigram_model, ['맛있는', '점심을'])}")

print("\n=== 감성 분류 ===")
data = {
    'reviews': [
        '이 영화 정말 좋아 최고야',
        '시간 아까운 쓰레기 영화',
        '배우들 연기가 너무 훌륭해요',
        '스토리가 지루하고 뻔하다',
        '인생 최고의 명작입니다 추천',
        '돈 주고 보기 아까운 졸작'
    ],
    'target': [1, 0, 1, 0, 1, 0]
}
df = pd.DataFrame(data)
N  = 3

models_info = {}
for label in df['target'].unique():
    corp = df[df['target'] == label]['reviews'].tolist()
    models_info[label] = build_ngram_model_with_smoothing(corp, N)

test_reviews = ['돈 주고 보기 아까운 졸작', '내가 본 영화중에 명작입니다']
for review in test_reviews:
    pred      = predict_sentiment(review, models_info, n=N)
    sentiment = "긍정" if pred == 1 else "부정"
    print(f"리뷰: '{review}' → {sentiment} (Class: {pred})")